# 03 · AlphaFold comparison and membrane context — PSMA / GCPII (Q04609)

**Milestone 4** (checklist item 7) and the **membrane part of item 15**, with a bridge to membrane MD.
Design and every decision: [`docs/M4_ALPHAFOLD_MEMBRANE.md`](../docs/M4_ALPHAFOLD_MEMBRANE.md).

The notebook does seven things:

1. **Gets predicted models, entirely inside Colab.**
   - The current **AlphaFold DB** model (a monomer) is fetched automatically.
   - **AlphaFold2-multimer is run here, on the Colab GPU, through ColabFold**, for the full-length PSMA **dimer**.
   - You can also add other ColabFold results or any PDB/mmCIF model.
2. **Numbers every model in UniProt numbering and checks its PAE indexing.** PAE rows are matched by residue key, never by position, and chain blocks are checked. This is a critical check.
3. **Reports confidence** per residue, region and domain: pLDDT, intra-region PAE and inter-domain PAE.
4. **Compares each model with the experimental templates** from notebook 01:
   - CA RMSD over the confident core and over all common residues;
   - TM-score and GDT-like fractions;
   - per-region RMSD;
   - active-site and Zn-ligand side-chain RMSD;
   - pLDDT calibration;
   - the entrance-lid state.
5. **Analyses multimers.** Interfaces are compared with the canonical interface from notebook 02, and a **confident-dimer rule** is applied: ipTM, interface PAE *and* agreement with experiment.
6. **Adds membrane context:**
   - UniProt versus DeepTMHMM topology;
   - membrane placement, constrained by the C2 axis for dimers;
   - membrane QC, ending in an *orientation reliability* call;
   - the construct context of the experimental structures.
7. **Writes membrane-ready inputs** (CHARMM-GUI / packmol-memgen) for the best-ranked model, with a readiness table. **These are not MD-ready.** Zn, glycans and protonation are Milestone 5.

Outputs go to `<ACC>_alphafold_membrane.xlsx`, `alphafold_tables/` and `md_membrane_inputs/`.

## 0 · Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
    subprocess.run(["git", "clone", "-q", REPO], check=True)
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if Path("../src/psma_pipeline").exists():
    os.chdir("..")
if IN_COLAB:
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[alphafold,membrane]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "freesasa"], check=False)  # optional; numpy fallback otherwise
sys.path.insert(0, str(Path("src").resolve()))
for _m in [m for m in list(sys.modules) if m == "psma_pipeline" or m.startswith("psma_pipeline.")]:
    del sys.modules[_m]
import psma_pipeline
print("project root:", Path.cwd(), "| psma_pipeline", psma_pipeline.__version__)

## 1 · Configuration and results from notebooks 01 and 02

The notebook reads three workbooks from the shared Google Drive folder:

| Workbook | From | Needed? |
|---|---|---|
| Survey | notebook 01 | **required**; it supplies the template order, construct ranges and active site |
| Assessment | notebook 02 part 1 | optional; it supplies the **canonical interface**. Without it, the dimer rule's third criterion is `NOT RUN`, so no dimer can be called *confident*. |
| Pockets / waters / variants | notebook 02 part 2 | optional; its `VariantStructureMapping` gains AlphaFold pLDDT |

> **Decision: everything runs in Colab.**
> - The monomer is the current **AlphaFold DB** model, fetched by accession.
> - The dimer is predicted here with **AlphaFold2-multimer v3** (ColabFold), so no external prediction service is needed.
>
> AlphaFold2 does not model Zn²⁺ or glycans. Both are listed as Milestone 5 blockers in the readiness table.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from psma_pipeline import load_config, setup_logging
from psma_pipeline.colab import storage_overrides
from psma_pipeline.assessment import load_survey_tables
from psma_pipeline.alphafold import run_alphafold, save_alphafold, detect_upload_kind, variant_alphafold_columns
from psma_pipeline.sanity import critical_failures

setup_logging()
USE_DRIVE = True
RUN_DEEPTMHMM = True        # DeepTMHMM via BioLib (needs network; NOT RUN and recorded if unavailable)
cfg = load_config(overrides={**(storage_overrides(USE_DRIVE) or {}),
                             "membrane": {"deeptmhmm": {"enabled": RUN_DEEPTMHMM}}})

SURVEY = cfg.output_file("survey_workbook")
ASSESSMENT = cfg.output_file("assessment_workbook")
POCKETS = cfg.output_file("pocket_workbook")
assert Path(SURVEY).exists(), f"{SURVEY} not found: run notebook 01 first (or upload it and set SURVEY)"
survey = load_survey_tables(SURVEY)
assessment = load_survey_tables(ASSESSMENT) if Path(ASSESSMENT).exists() else None
pockets = load_survey_tables(POCKETS) if Path(POCKETS).exists() else None
print("survey:", SURVEY, f"({len(survey['Summary'])} entries)")
print("assessment:", ASSESSMENT if assessment else "not found (canonical-interface criterion will be NOT RUN)")
print("pockets/variants:", POCKETS if pockets else "not found (variant AlphaFold columns skipped)")

## A · Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "Unit tests failed: fix before running the AlphaFold stage."

## B · Run AlphaFold2 in Colab (ColabFold)

**Runtime → Change runtime type → GPU** (a T4 is enough; an A100 or L4 is much faster).

The jobs are set in `alphafold.colabfold.jobs`. By default only the **full-length dimer** runs: 2 × 750 residues, AlphaFold2-multimer v3. It is needed for:
- the confident-dimer rule;
- the C2-constrained membrane placement.

**Settings:**

| Setting | Default | Notes |
|---|---|---|
| `NUM_MODELS` | 3 of AlphaFold2's 5 | Each takes roughly 15–40 min for the dimer on a T4 |
| `NUM_RECYCLE` | 6 | `None` = ColabFold's default (up to 20, with early stopping) |
| `TEMPLATES` | False | True uses PDB templates found by the MMseqs2 server |

> **Resumable.** Results go to the cache, which is Google Drive in Colab. If the runtime disconnects, re-run this section: finished models are kept and finished jobs are skipped.

The MSA is built by the ColabFold MMseqs2 server, which is called from this notebook. That is the same as ColabFold's own notebook, so no sequence leaves the Colab workflow by hand.

In [ ]:
RUN_ALPHAFOLD2 = True        # False: use AlphaFold DB (+ any previous ColabFold results) only
NUM_MODELS = 3
NUM_RECYCLE = 6
TEMPLATES = False
JOBS = {"dimer": {"enabled": True, "copies": 2},            # full-length homodimer (default)
        "monomer": {"enabled": False, "copies": 1},         # AlphaFold DB already supplies a monomer
        "ectodomain_dimer": {"enabled": False, "copies": 2, "start": 44}}

from psma_pipeline.colabfold_run import gpu_name, install_colabfold, run_colabfold_jobs
cfg = cfg.with_overrides({"alphafold": {"colabfold": {"num_models": NUM_MODELS, "num_recycle": NUM_RECYCLE,
                                                      "templates": TEMPLATES, "jobs": JOBS}}})
print("GPU:", gpu_name() or "NONE - switch the runtime to GPU before running AlphaFold2")
if RUN_ALPHAFOLD2:
    print("installing ColabFold ..." )
    assert install_colabfold(), "ColabFold installation failed (see the log above)"
    print("ColabFold ready")

In [ ]:
from psma_pipeline.status import IssueLog
from psma_pipeline.uniprot import uniprot_entry, uniprot_info
from psma_pipeline.cache import session_from_config
sess = session_from_config(cfg)
useq = uniprot_info(uniprot_entry(sess, cfg.accession, cfg.cache_dir, cfg.endpoint("uniprot_json")))[0]
af2_issues = IssueLog()
uploads = run_colabfold_jobs(cfg, useq, af2_issues) if RUN_ALPHAFOLD2 else []
display(af2_issues.to_frame())
for u in uploads:
    print(u["label"], "->", u["path"])
    print("   ", u["command"])

In [ ]:
# Optional: add other models already on disk or uploaded (ColabFold result zips, PDB/mmCIF files)
ADD_OTHER_MODELS = False
PLDDT_IN_B = False           # plain PDB/mmCIF files: B-factor column holds pLDDT
if ADD_OTHER_MODELS and IN_COLAB:
    from google.colab import files
    updir = cfg.cache_dir / "alphafold" / "uploads"
    updir.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        p = updir / name
        if not p.exists():
            p.write_bytes(data)          # uploads are never overwritten
        uploads.append({"kind": detect_upload_kind(p), "path": str(p), "label": Path(name).stem, "plddt_in_b": PLDDT_IN_B})
print(f"{len(uploads)} model set(s) besides AlphaFold DB")

## C · Run

The run imports and numbers the models and checks their PAE. It then loads the experimental references: the top `alphafold.compare.top_n` templates plus the lid calibration structures. After that it compares, analyses interfaces, places the membrane, ranks and writes the MD inputs.

> **Gate.** A critical failure stops the notebook:
> - `pae_indexing`: the PAE matrix does not match the residue index, or a chain block is broken;
> - `numbering_identity`: a target chain is below 90 % identity to UniProt.

In [ ]:
res = run_alphafold(cfg, survey, assessment, uploads=uploads, sess=sess, issues=af2_issues)
T = res.tables
checks = T["AlphaFoldChecks"]
display(checks)
crit = critical_failures(checks)
assert crit.empty, "critical AlphaFold check failed: see AlphaFoldChecks"
display(res.issues.to_frame())

## D · Confidence and comparison with experiment

pLDDT bands: ≥ 90 very high, 70–90 confident, 50–70 low, < 50 very low.

The **confident core** (pLDDT ≥ 70) is used for superposition, so disordered tails do not inflate the RMSD. A **negative** pLDDT–deviation Spearman means the model is *least* confident where it deviates most from experiment, which is what you want.

In [ ]:
cols = ["Model ID", "Source", "Model kind", "Version", "Mean pLDDT", "Mean pLDDT ectodomain", "Mean pLDDT TM",
        "Mean pLDDT active site", "pTM", "ipTM", "Best template", "Core CA RMSD to best template (Å)",
        "TM-score to best template", "Zn-ligand side-chain RMSD (Å)", "pLDDT–deviation Spearman",
        "Entrance lid state", "Dimer confidence", "Orientation reliability"]
display(T["AFModels"][[c for c in cols if c in T["AFModels"]]])

In [ ]:
rc = T["AFResidueConfidence"]
doms = cfg.regions.get("domains", {})
for mid, g in rc.groupby("Model ID"):
    g = g[g["Chain"] == g["Chain"].iloc[0]]
    fig, ax = plt.subplots(2, 1, figsize=(13, 4.2), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
    for lo, hi, c in [(90, 100, "#0053D6"), (70, 90, "#65CBF3"), (50, 70, "#FFDB13"), (0, 50, "#FF7D45")]:
        ax[0].axhspan(lo, hi, color=c, alpha=0.15, lw=0)
    ax[0].plot(g["UniProt number"], g["pLDDT"], color="k", lw=0.8, label="pLDDT")
    if "Deviation from best template (Å)" in g:
        a2 = ax[0].twinx()
        a2.plot(g["UniProt number"], g["Deviation from best template (Å)"], color="tab:red", lw=0.7, alpha=0.8)
        a2.set_ylabel("deviation (Å)", color="tab:red")
    ax[0].set(ylabel="pLDDT", ylim=(0, 100), title=f"{mid}: confidence and deviation from {g['Deviation reference'].iloc[0] if 'Deviation reference' in g else 'experiment'}")
    for k, (name, (a, b)) in enumerate(doms.items()):
        ax[1].barh(0, b - a + 1, left=a - 0.5, color=plt.cm.tab10(k % 10), height=0.8)
        ax[1].text((a + b) / 2, 0, name.split(" ")[0], ha="center", va="center", fontsize=7, color="white")
    ax[1].set(yticks=[], xlabel="UniProt residue")
    plt.tight_layout(); plt.show()

In [ ]:
m = next((x for x in res.models if x.pae is not None), None)
if m is not None:
    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    im = ax.imshow(m.pae, cmap="Greens_r", vmin=0, vmax=30)
    # chain and domain boundaries along the PAE index
    for k, (c, u) in enumerate(m.pae_index):
        if k and (m.pae_index[k - 1][0] != c or u in {a for a, _ in doms.values()}):
            ax.axhline(k - 0.5, color="tab:red" if m.pae_index[k - 1][0] != c else "grey", lw=0.6)
            ax.axvline(k - 0.5, color="tab:red" if m.pae_index[k - 1][0] != c else "grey", lw=0.6)
    plt.colorbar(im, ax=ax, label="expected position error (Å)")
    ax.set(title=f"PAE · {m.model_id} (red = chain, grey = domain boundaries)", xlabel="scored residue", ylabel="aligned residue")
    plt.tight_layout(); plt.show()
dp = T["AFDomainPAE"]
if len(dp):
    first = dp["Model ID"].iloc[0]
    d = dp[(dp["Model ID"] == first) & (dp["Chain (aligned)"] == dp["Chain (scored)"])]
    display(d.pivot_table(index="Domain (aligned)", columns="Domain (scored)", values="Mean PAE (Å)").round(1))

In [ ]:
display(T["AFvsExperimental"])
display(T["AFRegionRMSD"].pivot_table(index=["Model ID", "Reference PDB ID"], columns="Region",
                                      values="CA RMSD after core fit (Å)").round(2) if len(T["AFRegionRMSD"]) else "no region RMSD")
display(T["AFEntranceLid"])

## E · Dimers and the confident-dimer rule

| Call | Condition |
|---|---|
| **confident** | ipTM ≥ 0.8 **and** interface PAE ≤ 5 Å **and** Jaccard ≥ 0.5 to the experimental canonical interface |
| **low confidence** | ipTM < 0.6 **or** interface PAE > 10 Å |
| **uncertain** | everything else, including any criterion that is not available |

High predictor confidence alone is not enough: the predicted interface must also be the one seen in the crystal and cryo-EM structures.

In [ ]:
fi = T["AFInterfaces"]
if len(fi):
    display(fi[["Model ID", "Chain 1", "Chain 2", "Residue contacts", "BSA total (Å²)", "ipTM", "Interface PAE (Å)",
                "Canonical interface Jaccard", "Fraction of canonical interface recovered", "Dimer confidence",
                "Dimer confidence reason"]])
else:
    print("No multimer models: run the AlphaFold2 dimer job in section B.")
display(T["AFRankingTop"])

## F · Membrane context

PSMA is a **type II membrane protein**: a cytoplasmic tail, one signal-anchor TM helix and a large ectodomain. No experimental structure contains the TM helix (see *ConstructContext*), so the membrane anchor comes from prediction alone.

**Placement.** The membrane slab is fitted to the topology. **For dimers, the membrane normal is constrained to the two-fold axis**, because a symmetric dimer in a flat membrane must have its symmetry axis along the normal.

**Orientation reliability:**
- `unsupported`: the TM helix is not in the membrane, the ectodomain cuts through the membrane, or a glycosylation sequon lies below the outer plane;
- `weak`: the model does not *determine* the ectodomain's orientation (TM–ectodomain PAE > 10 Å), the TM pLDDT is < 70, or the C2 axis is tilted;
- `supported`: otherwise.

For PSMA, expect `weak` on the monomer: the stalk is flexible, and AlphaFold usually reports a high TM–ectodomain PAE. The AlphaFold2 dimer is the more informative model, because its two TM helices and its symmetry axis constrain the membrane.

In [ ]:
display(T["TopologyComparison"])
pc = ["Model ID", "Placement method", "TM in slab", "Loop side agreement", "Membrane thickness (Å)", "TM helix tilt (°)",
      "TM pLDDT", "TM-ectodomain PAE (Å)", "Ectodomain lowest atom above outer plane (Å)", "Ectodomain atoms in slab",
      "Sequons on wrong side", "Active-site height above membrane (Å)", "C2 axis to membrane normal (°)",
      "Orientation reliability", "Reliability reasons"]
display(T["MembranePlacement"][[c for c in pc if c in T["MembranePlacement"]]])
display(T["ConstructContext"].tail(1))

In [ ]:
# 3D view of the membrane-oriented MD-input model (membrane planes as dummy atoms)
md_dir = res.membrane.md_files.get("folder") if res.membrane else None
if md_dir:
    try:
        import py3Dmol
        v = py3Dmol.view(width=820, height=560)
        v.addModel(open(Path(md_dir) / "model_membrane_oriented.pdb").read(), "pdb")
        v.setStyle({"model": 0}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 40, "max": 95}}})
        v.addModel(open(Path(md_dir) / "membrane_planes_dum.pdb").read(), "pdb")
        v.setStyle({"model": 1, "elem": "N"}, {"sphere": {"radius": 0.5, "color": "#4c78a8", "opacity": 0.5}})
        v.setStyle({"model": 1, "elem": "O"}, {"sphere": {"radius": 0.5, "color": "#e15759", "opacity": 0.5}})
        v.rotate(-90, "x"); v.zoomTo(); v.show()
        print("coloured by pLDDT (red low → blue high); blue plane = cytoplasmic, red plane = outer")
    except ImportError:
        print("py3Dmol not installed: open the PDB files in PyMOL / ChimeraX")

## G · Membrane-ready inputs and MD readiness

The model is chosen as follows:
- the **top `membrane_dimer`** model, but only if its dimer call is **confident**;
- otherwise the top **`membrane_anchor`** monomer, recorded as a WARNING. The functional dimer can then be built on the canonical interface in Milestone 5.

> **These files are not MD-ready.** The readiness table lists every blocker, and its overall status is always `FAIL` in this milestone. The lipid composition is a **placeholder** (`membrane.builder` in the config): choose one appropriate to your system.

In [ ]:
display(T["MembraneMDReadiness"])
if md_dir:
    for f in sorted(Path(md_dir).iterdir()):
        print(f)
    print(open(Path(md_dir) / "packmol_memgen_command.sh").read())

## H · Variants: AlphaFold pLDDT (fills the Milestone 3 placeholder)

In [ ]:
if pockets and "VariantStructureMapping" in pockets and len(T["AFResidueConfidence"]):
    T["VariantAlphaFold"] = variant_alphafold_columns(pockets["VariantStructureMapping"], T["AFResidueConfidence"])
    display(T["VariantAlphaFold"][["Substitution", "Categories", "AlphaFold pLDDT", "AlphaFold status"]].head(30))
else:
    print("pockets/variants workbook not found: run notebook 02 part 2 to map variants")

## I · Save

In [ ]:
paths = save_alphafold(cfg, res)
for k, v in paths.items():
    print(f"{k}: {v}")

## Milestone 4 checklist

| Component | Status |
|---|---|
| AlphaFold DB import (current release; current and retired API field names) | ✔ |
| AlphaFold2 / AlphaFold2-multimer run inside Colab (ColabFold), resumable | ✔ |
| Import of other ColabFold results and any PDB/mmCIF model | ✔ |
| UniProt numbering by alignment (constructs, tags, mutations) | ✔ |
| PAE indexing by residue key; chain-boundary checks (critical) | ✔ |
| pLDDT/PAE per residue, region and domain | ✔ |
| Comparison with experimental templates: core RMSD, TM-score, GDT-like, regions, active site, calibration, lid | ✔ |
| Multimer interfaces and the confident-dimer rule | ✔ |
| AlphaFold rankings (ectodomain, active site, dimer, membrane anchor, membrane dimer) | ✔ |
| Membrane topology (UniProt / DeepTMHMM), C2-constrained placement, membrane QC, construct context | ✔ |
| Membrane-ready CHARMM-GUI / packmol-memgen inputs and readiness table (never READY) | ✔ |
| Variant AlphaFold pLDDT columns (Milestone 3 placeholder) | ✔ |